# Day 05 下午学生项目：电商用户多维分析

**小组编号：** Group 01
**成员：** 张三
**专题方向：** A

> 请只在标有 `TODO` 的区域填写代码，不要删除任务说明、检查点和反思题。


## 实验目标与提交要求

你需要完成：

1. 数据加载与验收；
2. 公共基础指标；
3. 一个单维专题分析；
4. 一个双维交叉分析；
5. 三个CSV报表；
6. 至少3条结论、1条限制和1项建议。

**重要边界：** 一行是一名用户；返现不是消费金额；相关不等于因果。


## 任务0：小组配置


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

GROUP_ID = "Group01"
MEMBERS = ["张三"]
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

ROOT = Path.cwd()
DATA_PATH = ROOT / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_student" / GROUP_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("小组：", GROUP_ID, MEMBERS)
print("专题：", TOPIC)
print("输入：", DATA_PATH)
print("输出：", OUTPUT_DIR)


### 检查点0

- [x] 已填写组号、成员和专题；
- [x] Notebook文件名包含组号；
- [x] 输出目录中的组号正确。


## 任务1：加载并验收数据（必做）


In [ ]:
# TODO 1：读取清洗后的CSV，变量名必须为df
df = pd.read_csv(DATA_PATH)

# TODO 2：输出shape、前5行和字段类型
print("数据形状：", df.shape)
print("\n前5行：")
display(df.head())
print("\n字段类型：")
display(df.dtypes)

# TODO 3：计算以下验收结果
validation = {
    "行数": df.shape[0],
    "列数": df.shape[1],
    "CustomerID重复数": df["CustomerID"].duplicated().sum(),
    "核心字段缺失数": df[['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice']].isnull().sum().sum(),
    "Churn取值": sorted(df["Churn"].unique()),
}
validation


In [ ]:
# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("检查点1通过")


**数据粒度：** 请用一句话填写：
一行数据代表一名用户的综合行为特征，包括用户基本信息、消费行为、偏好设置等维度。


## 任务2：公共基础指标（必做）


In [ ]:
# TODO：构建overall_metrics DataFrame，至少包含以下指标：
# 用户数、流失人数、流失率、平均订单数、订单数中位数、
# 平均优惠券数、平均返现、平均App时长、平均满意度、平均距上次下单天数
overall_metrics = pd.DataFrame({
    "指标": [
        "用户数", "流失人数", "流失率", "平均订单数", "订单数中位数",
        "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数",
        "注册设备数均值", "地址数均值", "投诉率", "平均订单涨幅"
    ],
    "数值": [
        df.shape[0],
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
        df["NumberOfDeviceRegistered"].mean(),
        df["NumberOfAddress"].mean(),
        df["Complain"].mean(),
        df["OrderAmountHikeFromlastYear"].mean()
    ]
})

display(overall_metrics)


In [ ]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# TODO：将下面变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率不正确"
print("检查点2通过")


## 任务3：单维专题分析（必做）


请选择一个专题：

- A：`TenureGroup` 用户生命周期；
- B：`Complain` 或 `SatisfactionScore` 服务体验；
- C：`PreferedOrderCat` 品类与订单；
- D：`PreferredPaymentMode` 支付与优惠；
- E：`CityTier` 或 `PreferredLoginDevice` 城市与设备。

最低要求：使用 `groupby + agg`，同时输出用户数和至少3项业务指标。


In [ ]:
# TODO：填写你的分组字段
segment_field = "TenureGroup"

# TODO：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=('CustomerID', 'count'),
    流失人数=('Churn', 'sum'),
    流失率=('Churn', 'mean'),
    平均订单数=('OrderCount', 'mean'),
    平均优惠券数=('CouponUsed', 'mean'),
    平均返现=('CashbackAmount', 'mean'),
    平均App时长=('HourSpendOnApp', 'mean'),
    平均满意度=('SatisfactionScore', 'mean'),
    投诉率=('Complain', 'mean')
)

# TODO：重置索引、排序并展示
segment_analysis = segment_analysis.reset_index()
segment_analysis = segment_analysis.sort_values('流失率', ascending=False)
display(segment_analysis)


In [ ]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")


### 专题分析记录

**数据现象：**
新用户的流失率最高，达到约53.5%；其次是0-6个月的用户，流失率约25.9%；而7-12个月的用户流失率降至约9.8%，13-24个月的用户流失率约6.5%，24个月以上的用户流失率为0%。

**可能解释：**
新用户可能因产品体验不符合预期或竞争对手吸引而流失，这是典型的新用户留存问题。随着用户使用时间增长，用户粘性逐渐增强，流失率持续下降。可能相关的因素包括用户习惯养成、会员权益积累等，但需进一步验证因果关系。


## 任务4：双维度交叉分析（必做）


In [ ]:
# TODO：从以下维度中选择两个
# TenureGroup、Complain、PreferedOrderCat、CityTier、PreferredLoginDevice
dim_1 = "TenureGroup"
dim_2 = "CityTier"

# TODO：按两个维度统计用户数、流失人数、流失率，以及至少一个行为指标
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=('CustomerID', 'count'),
    流失人数=('Churn', 'sum'),
    流失率=('Churn', 'mean'),
    平均订单数=('OrderCount', 'mean'),
    平均满意度=('SatisfactionScore', 'mean')
)

# TODO：新增"样本提示"列；用户数<30标记为"小样本"，否则为"可观察"
cross_analysis = cross_analysis.reset_index()
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(lambda x: "小样本" if x < 30 else "可观察")

# TODO：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values('流失率', ascending=False)
display(cross_analysis)


In [ ]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")


### 双维分析记录

**最值得关注的组合：**
新用户 + 2线城市的组合流失率最高（93.3%），但样本量仅15人为小样本；新用户 + 3线城市的组合流失率57.5%，样本量181人，是更具参考价值的高流失组合。

**该组合的样本量与流失率：**
新用户+3线城市组合样本量181人，流失率57.5%；新用户+1线城市组合样本量312人，流失率49.4%，均显著高于总体平均流失率16.8%。

**为什么不能直接下因果结论：**
数据只展示了相关性，无法排除其他混杂因素的影响。例如，3线城市的新用户可能面临产品适配性问题、物流体验较差、或本地竞争更激烈等因素，但这些都需要进一步的数据或实验来验证。


## 任务5：报表输出与回读验证（必做）


In [ ]:
# TODO：将三个表导出到OUTPUT_DIR
# 文件名必须为：overall_metrics.csv、segment_analysis.csv、cross_analysis.csv
# 要求：index=False，encoding="utf-8-sig"

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

# TODO：循环导出并重新读取；打印每个文件的shape
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    print(f"{filename}: 原始shape={table.shape}, 回读shape={reloaded.shape}")


In [ ]:
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")


## 任务6：结论、限制与建议（必做）


### 结论1

在新用户中，流失率指标为53.5%，与24个月以上用户（流失率0%）相比存在显著差异。对应证据表：segment_analysis.csv。

### 结论2

3线城市的新用户流失率最高，达到57.5%，高于1线城市新用户的49.4%，表明城市层级与用户生命周期存在交互效应，3线城市新用户是重点关注群体。对应证据表：cross_analysis.csv。

### 结论3

随着用户使用时间增长，平均返现金额呈现上升趋势，新用户平均返现为142.4元，而24个月以上用户平均返现达到222.3元，这可能与用户忠诚度奖励机制有关。对应证据表：segment_analysis.csv。

### 分析限制

缺少订单金额和订单日期等关键交易数据，因此无法计算GMV、客单价或用户消费的时间趋势。此外，数据为截面数据，无法观察用户在不同时间点的行为变化，难以建立因果关系。

### 运营建议与验证方式

建议针对3线城市的新用户制定专属的留存策略，例如提供本地化的促销活动、优化物流体验、增加新手引导等。验证方式：通过A/B测试对比策略实施前后的用户留存率变化，同时监测用户满意度和投诉率指标的改善情况。需要收集更多关于用户流失原因的定性数据（如问卷调研）来进一步验证策略有效性。


## 拓展任务（选做）


In [ ]:
# 可选方向：
# 1. 使用qcut构建订单活跃度分层；
# 2. 设计供第6天绘图使用的长表；
# 3. 对反直觉结果提出两种数据核查方法。

# TODO（选做）
# 使用qcut构建订单活跃度分层
df['OrderActivity'] = pd.qcut(df['OrderCount'], q=4, labels=['低活跃度', '较低活跃度', '较高活跃度', '高活跃度'])

# 按订单活跃度分层统计
activity_analysis = df.groupby('OrderActivity').agg(
    用户数=('CustomerID', 'count'),
    流失率=('Churn', 'mean'),
    平均满意度=('SatisfactionScore', 'mean'),
    平均返现=('CashbackAmount', 'mean')
).reset_index()

print("订单活跃度分层分析：")
display(activity_analysis)


## 提交前检查

- [x] 已填写组号、成员和专题；
- [x] 已重启内核并从头运行成功；
- [x] 所有比例表都包含样本量；
- [x] 三个CSV已导出并回读；
- [x] 至少3条结论可对应到具体表格；
- [x] 已写明分析限制和验证建议；
- [x] 没有把返现写成消费额，没有把相关写成因果。
